In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q chemprop

import torch
print("GPU available:", torch.cuda.is_available())

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.3/154.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.0/330.0 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 89.3 MB/s eta 0:00:00
GPU available: True


In [3]:
import pandas as pd, numpy as np

base = "/content/drive/MyDrive/NIR_Thesis"
clean = pd.read_csv(f"{base}/Data/my_data_clean.csv")

for s in ["train", "val", "test"]:
    clean[clean.split == s][["chrom", "solv", "emi_eV"]].to_csv(f"/content/cp_{s}.csv", index=False)

pd.read_csv("/content/cp_test.csv").head()

,chrom,solv,emi_eV
0,BrC1=CN2C[n+]3cc(Br)c4ccccc4c3N=C2c2ccccc21,CO,2.469801
1,C(#Cc1ccc2ccc3cccc4ccc1c2c34)c1ccccc1,Cc1ccccc1,3.170946
2,C(#Cc1ccc2ncsc2c1)c1ccc(N(c2ccccc2)c2ccccc2)cc1,Cc1ccccc1,2.896822
3,C(#Cc1cccc2nsnc12)c1ccc(-n2c3ccccc3c3ccccc32)cc1,C(#Cc1cccc2nsnc12)c1ccc(-n2c3ccccc3c3ccccc32)cc1,2.455129
4,C(#Cc1cccc2nsnc12)c1ccc(-n2c3ccccc3c3ccccc32)cc1,ClCCl,2.304535


In [4]:
!chemprop train \
  --data-path /content/cp_train.csv /content/cp_val.csv /content/cp_test.csv \
  --smiles-columns chrom solv \
  --target-columns emi_eV \
  --task-type regression \
  --epochs 40 --patience 10 --batch-size 64 \
  --message-hidden-dim 300 --depth 4 --ffn-num-layers 2 \
  --save-dir "/content/drive/MyDrive/NIR_Thesis/results/dmpnn_model"

Streaming output truncated to the last 5000 lines.
Epoch 28/39 ━━━━━━━━━━━╸━━━━ 165/224 0:00:15 •       10.92it/s v_num: 0.000     
                                     0:00:06                   train_loss_step: 
                                                               0.027 val_loss:  
                                                               0.242            
                                                               train_loss_epoch:
Epoch 28/39 ━━━━━━━━━━━╸━━━━ 166/224 0:00:15 •       10.92it/s v_num: 0.000     
                                     0:00:06                   train_loss_step: 
                                                               0.027 val_loss:  
                                                               0.242            
                                                               train_loss_epoch:
Epoch 28/39 ━━━━━━━━━━━╸━━━━ 167/224 0:00:15 •       10.93it/s v_num: 0.000     
                                     0:00:06              

In [6]:
!chemprop predict \
  --test-path /content/cp_test.csv \
  --smiles-columns chrom solv \
  --model-paths "/content/drive/MyDrive/NIR_Thesis/results/dmpnn_model" \
  --preds-path /content/dmpnn_test_preds.csv

2026-10-22T11:06:57 - INFO:chemprop.cli.main - Running in mode 'predict' with args: {'smiles_columns': ['chrom', 'solv'], 'reaction_columns': None, 'no_header_row': False, 'num_workers': 0, 'batch_size': 64, 'accelerator': 'auto', 'devices': 'auto', 'rxn_mode': 'REAC_DIFF', 'multi_hot_atom_featurizer_mode': 'V2', 'keep_h': False, 'add_h': False, 'ignore_stereo': False, 'reorder_atoms': False, 'molecule_featurizers': None, 'descriptors_path': None, 'descriptors_columns': None, 'no_descriptor_scaling': False, 'no_atom_feature_scaling': False, 'no_atom_descriptor_scaling': False, 'no_bond_feature_scaling': False, 'no_bond_descriptor_scaling': False, 'atom_features_path': None, 'atom_descriptors_path': None, 'bond_features_path': None, 'bond_descriptors_path': None, 'constraints_path': None, 'constraints_to_targets': None, 'use_cuikmolmaker_featurization': False, 'test_path': PosixPath('/content/cp_test.csv'), 'output': PosixPath('/content/dmpnn_test_preds.csv'), 'drop_extra_columns': Fals

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

preds = pd.read_csv("/content/dmpnn_test_preds.csv")
test = clean[clean.split == "test"].reset_index(drop=True)
assert (preds.chrom.values == test.chrom.values).all()   # rows line up with the test set

true = test.emi_nm.values
pred = 1239.84 / preds.emi_eV.values      # eV back to nm

print(f"D-MPNN  Test MAE: {mean_absolute_error(true, pred):.1f} nm   "
      f"RMSE: {mean_squared_error(true, pred)**0.5:.1f}   R²: {r2_score(true, pred):.3f}")
for lo, hi in [(0, 500), (500, 650), (650, 750), (750, 2000)]:
    m = (true >= lo) & (true < hi)
    print(f"{lo}-{hi} nm  n={m.sum():4d}  MAE={mean_absolute_error(true[m], pred[m]):5.1f}  "
          f"bias={np.mean(pred[m] - true[m]):6.1f}")

# save next to the XGBoost results
xgb = pd.read_csv(f"{base}/results/xgb_test_predictions.csv")
xgb["dmpnn_pred_nm"] = pred
xgb.to_csv(f"{base}/results/test_predictions_both.csv", index=False)

D-MPNN  Test MAE: 29.9 nm   RMSE: 42.2   R²: 0.789
0-500 nm  n= 952  MAE= 25.9  bias=   3.0
500-650 nm  n= 684  MAE= 31.0  bias=  -6.5
650-750 nm  n= 100  MAE= 58.2  bias= -33.3
750-2000 nm  n=  43  MAE= 35.6  bias= -30.2
